# Model Metrics Audit (Read-Only)

This notebook audits existing model artifacts and reports. It does **not** train models.

## Goals
- Load stage1/stage2 metrics from `reports/metrics`
- Compare key KPIs (accuracy, macro F1, attack recall, normal recall)
- Plot confusion matrices and per-class metrics when available
- Save a compact audit summary for reporting

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

ROOT = Path.cwd()

DATASET_CONFIGS = {
    "UNSW": {
        "stage1_dir": ROOT / "reports/metrics/unsw/two_stage/stage1",
        "stage2_dir": ROOT / "reports/metrics/unsw/two_stage/stage",
        "audit_dir": ROOT / "reports/metrics/unsw/two_stage/audit",
    },
    "CIC": {
        "stage1_dir": ROOT / "reports/metrics/cic/two_stage/stage1",
        "stage2_dir": ROOT / "reports/metrics/cic/two_stage/stage",
        "audit_dir": ROOT / "reports/metrics/cic/two_stage/audit",
    },
}

print("Setup loaded. Choose DATASET in next cell.")

In [ ]:
DATASET = "UNSW"  # "UNSW" or "CIC"
assert DATASET in DATASET_CONFIGS, f"Unsupported DATASET={DATASET}"

cfg = DATASET_CONFIGS[DATASET]
cfg["audit_dir"].mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET}")
print(f"Stage1: {cfg['stage1_dir']}")
print(f"Stage2: {cfg['stage2_dir']}")
print(f"Audit:  {cfg['audit_dir']}")

In [ ]:
def read_json(path: Path) -> dict | None:
    if not path.exists():
        return None
    with open(path) as f:
        return json.load(f)


def read_csv(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path)


def extract_summary(eval_summary: dict | None, stage: str) -> dict:
    if not eval_summary or "test" not in eval_summary:
        return {"stage": stage, "available": False}
    t = eval_summary["test"]
    return {
        "stage": stage,
        "available": True,
        "accuracy": t.get("accuracy", np.nan),
        "macro_f1": t.get("macro_f1", np.nan),
        "weighted_f1": t.get("weighted_f1", np.nan),
        "normal_recall": t.get("normal_recall", np.nan),
        "attack_recall": t.get("attack_recall", np.nan),
        "normal_precision": t.get("normal_precision", np.nan),
        "attack_precision": t.get("attack_precision", np.nan),
    }

In [ ]:
stage1_eval = read_json(cfg["stage1_dir"] / "eval_summary.json")
stage2_eval = read_json(cfg["stage2_dir"] / "eval_summary.json")

summary_df = pd.DataFrame([
    extract_summary(stage1_eval, "stage1_binary"),
    extract_summary(stage2_eval, "stage2_multiclass_filtered"),
])

summary_out = cfg["audit_dir"] / "model_audit_summary.csv"
summary_df.to_csv(summary_out, index=False)
display(summary_df)
print(f"Saved: {summary_out}")

In [ ]:
# Confusion matrix visualization (if available)
cm_stage1_path = cfg["stage1_dir"] / "confusion_matrix_test.csv"
cm_stage2_path = cfg["stage2_dir"] / "confusion_matrix_test.csv"

cm_stage1 = read_csv(cm_stage1_path)
cm_stage2 = read_csv(cm_stage2_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if cm_stage1 is not None:
    cm1 = cm_stage1.select_dtypes(include=[np.number]).to_numpy()
    sns.heatmap(cm1, annot=True, fmt=".0f", cmap="Blues", ax=axes[0])
    axes[0].set_title(f"{DATASET} Stage1 Confusion Matrix")
else:
    axes[0].text(0.5, 0.5, "No stage1 confusion matrix", ha="center", va="center")
    axes[0].set_axis_off()

if cm_stage2 is not None:
    cm2 = cm_stage2.select_dtypes(include=[np.number]).to_numpy()
    sns.heatmap(cm2, annot=True, fmt=".0f", cmap="Greens", ax=axes[1])
    axes[1].set_title(f"{DATASET} Stage2 Confusion Matrix")
else:
    axes[1].text(0.5, 0.5, "No stage2 confusion matrix", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
fig_out = cfg["audit_dir"] / "confusion_matrices.png"
plt.savefig(fig_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_out}")

In [ ]:
# Per-class F1/Recall (from stage2 classification report)
stage2_report_path = cfg["stage2_dir"] / "classification_report.csv"
stage2_report = read_csv(stage2_report_path)

if stage2_report is not None and "Unnamed: 0" in stage2_report.columns:
    stage2_report = stage2_report.rename(columns={"Unnamed: 0": "class"})

if stage2_report is not None and {"class", "f1-score", "recall"}.issubset(stage2_report.columns):
    filtered = stage2_report[
        ~stage2_report["class"].isin(["accuracy", "macro avg", "weighted avg"])
    ].copy()

    melted = filtered.melt(id_vars=["class"], value_vars=["f1-score", "recall"], var_name="metric", value_name="value")

    plt.figure(figsize=(12, 6))
    sns.barplot(data=melted.sort_values("value", ascending=False), x="value", y="class", hue="metric")
    plt.title(f"{DATASET} Stage2 Per-Class Recall and F1")
    plt.xlabel("Score")
    plt.ylabel("Class")
    plt.tight_layout()
    fig_out = cfg["audit_dir"] / "stage2_per_class_recall_f1.png"
    plt.savefig(fig_out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_out}")
else:
    print(f"Stage2 classification report missing or incompatible: {stage2_report_path}")

## Notes

- This notebook is intentionally read-only with respect to models.
- If a metric file is missing, cells continue gracefully and report what is unavailable.
- Use this as a reproducible reporting layer after training scripts finish.